In [ ]:
import polars as pl
from zh import ETH_USDT, SHARPE, scan

data = scan("1m").filter(symbol=ETH_USDT).collect(engine="streaming")

根据其他研究，可以认为持仓时间过短，噪声过大。

In [ ]:
def backtest_single_asset(
    data: pl.LazyFrame | pl.DataFrame,
    symbol: str,
    start_date,
    end_date,
    entry_z: float = 2.0,
    fee_bps: float = 0.0,
    window_size: int = 14,
    vol_threshold: float = 0.001,
) -> pl.LazyFrame:
    """
    Backtest a mean reversion strategy on a single asset with 1-minute hold period.

    Parameters:
    -----------
    data : polars.LazyFrame | polars.Dataframe
        close data with columns: symbol, open_time, close
    symbol : str
        Symbol to trade (e.g., 'ETHUSDT')
    start_date : datetime
        Start date for backtest
    end_date : datetime
        End date for backtest
    entry_z : float
        Z-score threshold for entry (default: 2.0)
    fee_bps : float
        Fee in basis points (default: 0)
    window_size : int
        Rolling window size for calculations (default: 14)
    """

    fee_rate = fee_bps / 1e4

    # Prepare data
    asset_data = (
        data.lazy()
        .filter(pl.col("symbol") == symbol)
        .select(
            "open_time",
            "open",
            "close",
            pl.col("close").log().alias("log_close"),
        )
        .filter((pl.col("open_time") >= start_date) & (pl.col("open_time") <= end_date))
        .sort("open_time")
    )

    # Backtest
    bt = (
        asset_data.with_columns(
            pl.col("log_close")
            .rolling_mean(window_size=window_size)
            .shift(1)
            .alias("rolling_mean"),
            pl.col("log_close")
            .rolling_std(window_size=window_size)
            .shift(1)
            .alias("rolling_std"),
        )
        .with_columns(
            (
                (pl.col("log_close") - pl.col("rolling_mean")) / pl.col("rolling_std")
            ).alias("zscore")
        )
        .with_columns(
            (pl.col("close") / pl.col("open") - 1).alias("returns"),
            pl.col("zscore").shift(1).alias("zscore_lag"),
        )
        .filter(pl.col("returns").is_not_null())
        .with_columns(
            # Mean reversion: buy when zscore is low (oversold), sell when high (overbought)
            pl.when(
                (
                    (pl.col("zscore_lag") <= -entry_z)
                    & (pl.col("close").pct_change().rolling_std(15) > vol_threshold)
                )
            )
            .then(1)  # Long when oversold
            .when(
                (pl.col("zscore_lag") >= entry_z)
                & (pl.col("close").pct_change().rolling_std(15) > vol_threshold)
            )
            .then(-1)  # Short when overbought
            .otherwise(0)
            .alias("pos")
        )
        .with_columns(
            pl.col("pos").shift(1).fill_null(0).alias("pos_prev"),
        )
        .with_columns(
            (pl.col("pos") * pl.col("returns")).alias("pnl_gross"),
            (pl.col("pos") - pl.col("pos_prev")).abs().alias("turnover"),
        )
        .with_columns((fee_rate * pl.col("turnover")).alias("fees"))
        .with_columns((pl.col("pnl_gross") - pl.col("fees")).alias("ret"))
        .fill_null(0)
    )

    return bt

In [ ]:
# Test pure ETH strategy
backtest_single_asset(
    data,
    symbol="ETHUSDT",
    start_date=pl.datetime(2023, 1, 1, time_zone="UTC"),
    end_date=pl.datetime(2025, 12, 31, time_zone="UTC"),
    entry_z=2.0,
    window_size=14,
    fee_bps=0.0,
    vol_threshold=0.0,
).group_by_dynamic(pl.col("open_time"), every="1d").agg(
    pl.col("ret").sum()
).with_columns(pl.col("ret").cum_sum()).collect().plot.line(x="open_time", y="ret")

In [ ]:
# Test pure ETH strategy
backtest_single_asset(
    data,
    symbol="ETHUSDT",
    start_date=pl.datetime(2023, 1, 1, time_zone="UTC"),
    end_date=pl.datetime(2025, 12, 31, time_zone="UTC"),
    entry_z=2.0,
    window_size=14,
    fee_bps=0.0,
    vol_threshold=0.000,
).select(sharpe=SHARPE).collect()